# Organisational Capability Configurations in UK SMEs
## Implications for Secure AI Adoption and Cyber Resilience

**Frida Alejandra Ibarra Zavaleta**
MA981-7-FY MSc Dissertation, University of Essex
School of Mathematics, Statistics and Actuarial Science

---

### What this notebook is

This notebook is the executable companion to the dissertation. It reproduces
every quantitative result reported there, and it is organised to follow the same
chapters, so that a reader can move between the two without translation:

| Notebook part | Dissertation | Question it answers |
|---|---|---|
| **Part 1** | Ch. 1 | What problem is this, and why does it matter? |
| **Part 2** | Ch. 4 | What do the data contain, and how were the indicators built? |
| **Part 3** | Ch. 3 | Why these methods rather than the obvious alternatives? |
| **Part 4** | Ch. 5 | What are the results? |
| **Part 5** | Ch. 3.8 | Do the results survive the checks? |
| **Part 6** | Ch. 5-6 | What do they mean, and what follows? |

Part 3 is the part a reader short of time should not skip. The methodological
choices in this study are not conventional defaults; each is justified below
against a named alternative, on evidence generated from these data.

### What the analysis can and cannot establish

Both sources publish population-weighted aggregate rates: four employment
size bands and six priority sectors. There are no record-level data. Every result
below is a statement about band-level or sector-level rates, never about
individual firms, and inference is applied only where the published tables supply
an effective base size to apply it to. The BICS release publishes no bases at
all, so the sectoral analysis is descriptive throughout.

All pseudo-random processes are seeded at 42. The notebook is deterministic and
runs top to bottom in under a minute.

In [1]:
import sys, os, json
sys.path.insert(0, "analysis")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import load_data as ld
import stats_core as sc
import model_selection as ms

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 3)

BICS = "data/wave147dsitaiadoptionrequest.xlsx"
CSBS = "data/DSIT_CSBS_2025-2026_tables.xlsx"
SEED = 42

print("python     :", sys.version.split()[0])
for m in (np, pd):
    print(f"{m.__name__:11s}:", m.__version__)
import scipy, sklearn, matplotlib
for m in (scipy, sklearn, matplotlib):
    print(f"{m.__name__:11s}:", m.__version__)
print("\nsources present:", os.path.exists(BICS), os.path.exists(CSBS))

python     : 3.13.5
numpy      : 2.1.3
pandas     : 2.2.3
scipy      : 1.15.3
sklearn    : 1.6.1
matplotlib : 3.10.0

sources present: True True


---
# Part 1. Motivation and research questions

### The problem

UK SMEs are being asked to adopt artificial intelligence and to strengthen their
cyber defences at the same time. Policy treats these as separate agendas. They
are not separable in practice: adopting AI widens the attack surface, and cyber
defence increasingly depends on AI. A third of UK businesses reported using at
least one AI technology in December 2025; 43% reported a breach or attack in the
preceding twelve months.

What nobody has established at national scale is the relationship between the
two inside the firm. That matters because the standard diagnosis, that SMEs
lack cyber governance in general, implies one kind of intervention. If instead
they hold general cyber governance and have simply not extended it to AI, the
implied intervention is entirely different: narrower, cheaper, and targeted.

### The three research questions

| | Question | Source | Inference available? |
|---|---|---|---|
| **RQ1** | Patterns of AI adoption by sector and technology type | ONS BICS Wave 147 | No: the release publishes no bases |
| **RQ2** | Cyber capability, governance and breach exposure by firm size | DSIT CSBS 2025/26 | Yes: effective bases published |
| **RQ3** | AI-specific vs general cyber governance, compared like-for-like | CSBS 2025/26 | Yes |

### The hypotheses

- **H1.** Configurations in which AI adoption advances without a corresponding
  extension of cybersecurity governance to AI-specific risk are common among UK
  SMEs, and this shortfall does not close as general governance improves with
  firm size.
- **H2.** AI and cyber capability indicators cohere as configurations that
  separate by organisational scale, rather than varying independently.

### The methodological problem this study had to solve

The published tables invite a comparison between the AI *adoption* rate and a
cyber *governance* composite. That comparison produces a clean, confident and
misleading answer, because it sets two different kinds of quantity against one
another. Part 4 reproduces it, shows why it fails, and substitutes a like-for-like
comparison that reverses the diagnosis.

---
# Part 2. Data description and preprocessing
### (Dissertation Chapter 4)

Two official UK releases, with incompatible disaggregation. BICS reports AI
adoption *by sector* with no size breakdown; CSBS reports *by size band* with
no sector breakdown. No sector-by-size cell is observable anywhere, which is why
RQ1 and RQ2 are answered as two separate pictures rather than one combined
analysis. This is a property of the published tables, not a choice.

### Preprocessing decisions, and why each was made

| Step | Decision | Reason |
|---|---|---|
| Extraction | By published row label, never by cell reference | A layout change raises an error instead of silently returning the wrong row |
| Suppressed cells | Carried as `NaN`, never imputed | Suppression concentrates in small sectors and rare categories; imputing would fabricate the most load-bearing values |
| Base sizes | Effective base, not respondent count | Design weighting inflates variance; using respondent counts would overstate precision by roughly 15-25% |
| Normalisation | None | The published rates are already proportions on a common scale |
| Dimensionality reduction | None | With four observations no latent structure is identifiable |

In [2]:
# Effective base vs respondent count: the difference is not cosmetic.
ind = ld.build_indicator_frame(CSBS)

base_cmp = ind[["size_band", "respondents_all", "n_all", "n_ai"]].copy()
base_cmp["design_effect"] = (base_cmp["respondents_all"] / base_cmp["n_all"]).round(2)
base_cmp["precision_lost_pct"] = (
    (1 - np.sqrt(base_cmp["n_all"] / base_cmp["respondents_all"])) * 100).round(1)
display(base_cmp)

print("Using respondent counts instead of effective bases would understate every")
print("standard error in this study by the percentages in the last column.")

,size_band,respondents_all,n_all,n_ai,design_effect,precision_lost_pct
0,Micro,1154.0,985.0,273.0,1.17,7.6
1,Small,507.0,441.0,156.0,1.15,6.7
2,Medium,296.0,259.0,143.0,1.14,6.5
3,Large,155.0,131.0,85.0,1.18,8.1


Using respondent counts instead of effective bases would understate every
standard error in this study by the percentages in the last column.


### Indicator construction (feature engineering)

Seven published rates, three derived quantities. The survey base of each is
tracked explicitly, because two of the indicators are not measured on the
same population, and the whole argument of RQ3 turns on that fact.

| Symbol | Definition | Source | Base |
|---|---|---|---|
| $T_s$ | Cyber training in last 12 months | Table 19 | All organisations |
| $B_s$ | Board member responsible for cyber | Table 9 | All organisations |
| $P_s$ | Formal cyber policy | Table 9 | All organisations |
| $A_s$ | Has adopted some AI tools | Table 22 | All organisations |
| $R_s$ | Any breach or attack in last 12 months | Table 28 | All organisations |
| $V_s$ | AI-specific cyber practices in place | Table 23 | **Organisations using or considering AI** |

Derived:

$$C_s = \frac{T_s + B_s + P_s}{3} \qquad
  G_s = A_s - C_s \qquad
  D_s = V_s - C_s \qquad
  U_s = 1 - V_s$$

$C_s$ is the cyber governance composite. $G_s$ is the naive gap, which Part 4
shows to be construct-invalid. $D_s$ is the AI-specific governance differential,
the construct-valid replacement. $U_s$ is unmanaged AI exposure.

In [3]:
display(ind[["size_band", "n_all", "n_ai", "T", "B", "P", "A", "V", "R",
             "C", "G", "D", "U"]].round(4))

print("\nBase heterogeneity, the single most important fact in this table:")
for _, r in ind.iterrows():
    print(f"  {r['size_band']:7s}  C_s rests on n={int(r['n_all']):4d} (all orgs)"
          f"   |   V_s rests on n={int(r['n_ai']):4d} (AI-engaged only)")

,size_band,n_all,n_ai,T,B,P,A,V,R,C,G,D,U
0,Micro,985.0,273.0,0.144,0.288,0.311,0.201,0.215,0.417,0.248,-0.047,-0.034,0.785
1,Small,441.0,156.0,0.331,0.369,0.522,0.243,0.273,0.463,0.407,-0.164,-0.134,0.727
2,Medium,259.0,143.0,0.538,0.516,0.747,0.393,0.456,0.648,0.600,-0.207,-0.145,0.544
3,Large,131.0,85.0,0.844,0.680,0.891,0.447,0.657,0.685,0.805,-0.358,-0.147,0.343



Base heterogeneity, the single most important fact in this table:
  Micro    C_s rests on n= 985 (all orgs)   |   V_s rests on n= 273 (AI-engaged only)
  Small    C_s rests on n= 441 (all orgs)   |   V_s rests on n= 156 (AI-engaged only)
  Medium   C_s rests on n= 259 (all orgs)   |   V_s rests on n= 143 (AI-engaged only)
  Large    C_s rests on n= 131 (all orgs)   |   V_s rests on n=  85 (AI-engaged only)


### Exploratory analysis: the three facts that shape everything downstream

The dissertation devotes a long section to exploratory analysis because three of
its analytical decisions depend on what that analysis establishes. Those three
facts are reproduced here.

In [4]:
# FACT 1. The deficit is asymmetric: narrow on purchasable controls, wide on
# practised ones. This is why a governance composite is the right construct and
# a controls count is the wrong one.
t15 = ld.load_csbs_table(CSBS, "Table15")

rows = []
for label, kind in [
    ("Up-to-date malware protection", "purchasable"),
    ("Firewalls that cover your entire IT network, as well as individual devices", "purchasable"),
    ("Backing up data securely via a cloud service", "purchasable"),
    ("Any monitoring of user activity", "practised"),
    ("Separate WiFi networks for staff and for visitors", "practised")]:
    s = t15.rate_series(label)
    r = sc.two_proportion_test(s[0], ind["n_all"][0], s[2], ind["n_all"][2])
    rows.append({"control": label[:46], "kind": kind,
                 "micro_pct": round(s[0]*100, 1), "medium_pct": round(s[2]*100, 1),
                 "ratio_micro_to_medium": round(s[0]/s[2], 2),
                 "cohens_h": round(r.cohens_h, 2), "magnitude": r.h_magnitude})

r = sc.two_proportion_test(ind["T"][0], ind["n_all"][0], ind["T"][2], ind["n_all"][2])
rows.append({"control": "Cyber security training", "kind": "practised",
             "micro_pct": round(ind["T"][0]*100, 1),
             "medium_pct": round(ind["T"][2]*100, 1),
             "ratio_micro_to_medium": round(ind["T"][0]/ind["T"][2], 2),
             "cohens_h": round(r.cohens_h, 2), "magnitude": r.h_magnitude})

asym = pd.DataFrame(rows)
display(asym)
print("Mean |h|, purchasable :",
      round(asym.loc[asym.kind == "purchasable", "cohens_h"].abs().mean(), 2))
print("Mean |h|, practised   :",
      round(asym.loc[asym.kind == "practised", "cohens_h"].abs().mean(), 2))

,control,kind,micro_pct,medium_pct,ratio_micro_to_medium,cohens_h,magnitude
0,Up-to-date malware protection,purchasable,80.2,90.9,0.88,-0.31,small
1,"Firewalls that cover your entire IT network, a",purchasable,72.0,89.5,0.81,-0.45,small
2,Backing up data securely via a cloud service,purchasable,72.7,84.0,0.87,-0.28,small
3,Any monitoring of user activity,practised,28.9,68.1,0.42,-0.81,large
4,Separate WiFi networks for staff and for visit,practised,33.5,80.3,0.42,-0.99,large
5,Cyber security training,practised,14.4,53.8,0.27,-0.87,large


Mean |h|, purchasable : 0.35
Mean |h|, practised   : 0.89


In [5]:
# FACT 2. Breach prevalence RISES with capability. Read naively this is the wrong
# sign for a protective effect; read correctly it says better-instrumented
# organisations notice more of what happens to them. Consequence: R_s cannot
# serve as an outcome variable anywhere in this study.
monitoring = t15.rate_series("Any monitoring of user activity")

for name, x in [("monitoring vs breach", monitoring),
                ("composite C_s vs breach", ind["C"].tolist())]:
    out = sc.spearman_exact(x, ind["R"].tolist())
    print(f"{name:26s} rho = {out['rho']:+.2f}   exact p = "
          f"{out['p_exact_two_sided']:.3f}   "
          f"(smallest attainable p at n=4: {out['min_attainable_p']:.3f})")

print("\nBreach rate by band:",
      [f"{v*100:.1f}%" for v in ind['R']])
print("Composite  by band:",
      [f"{v*100:.1f}%" for v in ind['C']])
print("\nNeither coefficient can reach conventional significance at n = 4, and")
print("neither is offered as evidence of association. The point is the SIGN.")

monitoring vs breach       rho = +1.00   exact p = 0.083   (smallest attainable p at n=4: 0.083)
composite C_s vs breach    rho = +1.00   exact p = 0.083   (smallest attainable p at n=4: 0.083)

Breach rate by band: ['41.7%', '46.3%', '64.8%', '68.5%']
Composite  by band: ['24.8%', '40.7%', '60.0%', '80.5%']

Neither coefficient can reach conventional significance at n = 4, and
neither is offered as evidence of association. The point is the SIGN.


In [6]:
# FACT 3. The incidents SMEs actually suffer are overwhelmingly human-mediated,
# while their investment is concentrated in technical controls. The right table
# here is Table 34, which asks about the MOST DISRUPTIVE breach: it is asked of
# organisations that can identify a single such incident, so the categories are
# mutually exclusive and the shares are interpretable as a composition.
t34 = ld.load_csbs_table(CSBS, "Table34")
phishing = t34.rate_series(
    "Phishing attacks, i.e. staff receiving fraudulent emails or arriving at "
    "fraudulent websites")
impersonation = t34.rate_series(
    "People impersonating, in emails or online, your organisation or your staff")

human = pd.DataFrame({
    "size_band": ld.SIZE_LABELS,
    "n_eff": [int(v) for v in t34.bases()],
    "phishing_pct": [round(v*100, 1) for v in phishing],
    "impersonation_pct": [round(v*100, 1) for v in impersonation],
})
human["human_mediated_pct"] = (human["phishing_pct"]
                               + human["impersonation_pct"]).round(1)
display(human)

sme = human[human.size_band != "Large"]["human_mediated_pct"]
print(f"Across the three SME bands, social engineering accounts for "
      f"{sme.min():.0f}% to {sme.max():.0f}% of the MOST DISRUPTIVE incident.")
print("\nThe controls that address social engineering are practised, not")
print("purchased: training, monitoring, reporting routines. Fact 1 showed those")
print("are exactly the controls where the SME deficit is widest. That mismatch")
print("between where the exposure is and where the investment goes is the single")
print("most important fact in the exploratory analysis.")

,size_band,n_eff,phishing_pct,impersonation_pct,human_mediated_pct
0,Micro,388,69.8,10.9,80.7
1,Small,200,67.7,18.1,85.8
2,Medium,160,58.8,25.8,84.6
3,Large,83,53.5,30.6,84.1


Across the three SME bands, social engineering accounts for 81% to 86% of the MOST DISRUPTIVE incident.

The controls that address social engineering are practised, not
purchased: training, monitoring, reporting routines. Fact 1 showed those
are exactly the controls where the SME deficit is widest. That mismatch
between where the exposure is and where the investment goes is the single
most important fact in the exploratory analysis.


---
# Part 3. Methodology: why these methods and not the alternatives
### (Dissertation Chapter 3)

This is the part that answers the question *"why did you choose that?"*

Terminology first. "Model selection" in a data science dissertation
normally means choosing among supervised learners. That is not the problem here.
Supervised learning needs records, a target and a
held-out set; this study has twenty aggregate rates on four bands and no outcome
that could serve as a label without committing an ecological fallacy. Fitting a
classifier to four observations would produce a model that is perfectly accurate,
entirely uninformative, and impossible to validate.

The methodological question that actually arises is harder at these sample sizes:
*given a published proportion and its effective base, which estimator states the
uncertainty honestly, and which procedure compares two of them without
manufacturing significance?*

Four decisions follow. Each is settled below on evidence rather than convention.

### 3.1 Which interval estimator?

Five candidates are in common use. The question is not which is best in general
but which is adequate at the combinations of rate and base this study actually
reports, which run from 1.3% to 89.1% on bases from 85 to 985.

Five anchor points were taken from the analysis itself, not invented for the
demonstration. At each, 20,000 binomial samples are drawn at the observed rate
and the empirical coverage of each nominal 95% interval is recorded. An interval
claiming 95% confidence should contain the true value 95% of the time.

In [7]:
anchors = ms.interval_anchors(ind)
for a in anchors:
    print(f"  {a['label']:26s} p={a['p']*100:5.1f}%  n={int(a['n']):4d}   ({a['why']})")

coverage = ms.run_coverage(anchors)
print("\nEmpirical coverage of a nominal 95% interval (%):")
display(coverage.pivot(index="method", columns="anchor", values="coverage_pct"))

print("Mean interval width (pp):")
display(coverage.pivot(index="method", columns="anchor", values="mean_width_pp"))

  AI adoption, medium        p= 39.3%  n= 259   (mid rate, mid base)
  AI practices, large        p= 65.7%  n=  85   (mid rate, smallest base in the study)
  Formal policy, large       p= 89.1%  n= 131   (high rate near the upper boundary)
  Ransomware, micro          p=  1.3%  n= 985   (very low rate, largest base)
  External hacking, large    p=  2.7%  n= 131   (very low rate, small base: the boundary case)

Empirical coverage of a nominal 95% interval (%):


anchor,"AI adoption, medium","AI practices, large","External hacking, large","Formal policy, large","Ransomware, micro"
method,,,,,
Agresti-Coull,95.22,94.88,97.50,95.15,96.19
Clopper-Pearson,95.22,96.01,97.50,96.34,96.10
Jeffreys,95.22,94.88,94.58,95.15,94.98
Wald,95.22,94.38,86.46,94.06,94.26
Wilson,95.22,94.88,97.50,95.15,94.98


Mean interval width (pp):


anchor,"AI adoption, medium","AI practices, large","External hacking, large","Formal policy, large","Ransomware, micro"
method,,,,,
Agresti-Coull,11.79,19.71,6.47,10.90,1.52
Clopper-Pearson,12.20,20.92,6.25,11.31,1.54
Jeffreys,11.82,19.78,5.51,10.56,1.44
Wald,11.87,20.04,5.25,10.57,1.42
Wilson,11.79,19.66,5.92,10.67,1.47


In [8]:
widths = ms.run_interval_comparison(anchors)

print("Does any candidate produce an INADMISSIBLE interval, i.e. one that")
print("extends outside [0, 1]?\n")
display(widths.groupby("method")["admissible"].all().rename("always_admissible"))

print("\nThe boundary case, where the choice actually bites:")
display(widths[widths.anchor == "External hacking, large"]
        [["method", "p_pct", "n_eff", "low_pct", "high_pct", "width_pp", "admissible"]])

Does any candidate produce an INADMISSIBLE interval, i.e. one that
extends outside [0, 1]?



method
Agresti-Coull       True
Clopper-Pearson     True
Jeffreys            True
Wald               False
Wilson              True
Name: always_admissible, dtype: bool


The boundary case, where the choice actually bites:


,method,p_pct,n_eff,low_pct,high_pct,width_pp,admissible
20,Wald,2.7,131,-0.10,5.42,5.51,False
21,Wilson,2.7,131,0.98,7.04,6.06,True
22,Clopper-Pearson,2.7,131,0.47,6.55,6.07,True
23,Agresti-Coull,2.7,131,0.70,7.32,6.62,True
24,Jeffreys,2.7,131,0.83,6.53,5.70,True


**Decision: Wilson.** The reasoning, in the order the evidence supports it:

1. **In the middle of the range the choice does not matter.** At
   $A_{\text{Medium}}$ all five estimators return 95.22% coverage and widths
   within 0.4pp of one another. Anyone analysing only mid-range rates could use
   any of them.
2. **At the boundary Wald fails.** For a rate of 2.7% on a base of 131, the Wald
   interval covers the true value in 86.5% of samples while claiming 95%, and
   its lower limit at the observed rate is -0.10%, a negative prevalence.
   That is not a rounding artefact; it is what happens to an interval that is not
   constrained to the parameter space.
3. **Among the survivors, Wilson is the efficient choice.** Clopper-Pearson buys
   its coverage guarantee with width and is widest at four of five anchors.
   Agresti-Coull is wider than Wilson at every boundary anchor and identical
   elsewhere, so it is dominated. Jeffreys is competitive and would have been
   defensible, but requires a prior this study would then have to defend.

The same logic carries to differences: the Newcombe hybrid-score interval is
built from two Wilson intervals, so it inherits their boundary behaviour. Using
Wilson for single proportions and then a normal approximation for their
difference would be internally inconsistent.

### 3.2 Which multiplicity correction?

Thirty pairwise size-band contrasts are run across five indicators. Under a
global null with no correction, the probability of at least one false rejection
at $\alpha = 0.05$ is $1 - 0.95^{30}$. Reporting thirty uncorrected $p$-values
would be indefensible. Four procedures are compared on the actual family.

In [9]:
family = ms.build_contrast_family(ind)
mult, mult_summary, mult_detail = ms.run_multiplicity(family)

print(f"FWER under a global null, uncorrected: "
      f"{mult['fwer_under_global_null']*100:.1f}%\n")
display(mult_summary)

changed = mult_detail[mult_detail["reject_holm"] != (mult_detail["p_value"] < 0.05)]
print(f"\nContrasts whose status CHANGES under Holm: {len(changed)}")
if len(changed):
    display(changed[["label", "p_value", "p_holm", "reject_holm"]])

print("\nContrasts that fail to survive Holm:")
display(mult_detail[~mult_detail["reject_holm"]]
        [["label", "diff_pp", "p_value", "p_holm", "cohens_h"]].round(4))

FWER under a global null, uncorrected: 78.5%



,procedure,controls,rejections,of_n_tests
0,Uncorrected,Nothing,26,30
1,Bonferroni,FWER (single-step),24,30
2,Holm-Bonferroni,FWER (step-down),26,30
3,Benjamini-Hochberg,FDR,26,30



Contrasts whose status CHANGES under Holm: 0

Contrasts that fail to survive Holm:


,label,diff_pp,p_value,p_holm,cohens_h
18,AI adoption: Micro vs Small,-4.14,0.078,0.313,-0.100
23,AI adoption: Medium vs Large,-5.40,0.306,0.612,-0.110
24,Any breach: Micro vs Small,-4.63,0.103,0.313,-0.093
29,Any breach: Medium vs Large,-3.76,0.459,0.612,-0.080


**Decision: Holm-Bonferroni.**

- It dominates plain Bonferroni: same error rate, same assumptions, two more
  rejections. That is a free gain, not a trade-off.
- Benjamini-Hochberg rejects the same 26 hypotheses here, but it controls the
  false discovery rate rather than the family-wise error rate, and it needs an
  assumption about the dependence structure that cannot be checked from published
  tables where the same respondents answer several questions. When two procedures
  agree, prefer the one making the weaker assumption.

Note what the comparison reveals: correction changes nothing substantive,
because the size gradients here are large relative to their standard errors. That
is reassuring, and it is reported precisely because a reader has no reason to
take it on trust. The two contrasts that fail are exactly the two the dissertation
argues should fail: micro vs small on breach prevalence and on AI adoption.

### 3.3 Which effect size, and why report one at all?

Because significance and magnitude separate at these bases. At $n_{\text{eff}} =
985$ a three-point difference can be significant and trivial; at $n_{\text{eff}} =
85$ a fifteen-point difference can be decisive and fragile. A raw percentage-point
difference is also not comparable across the range of $p$: 2% to 4% doubles the
prevalence, 48% to 50% barely moves it, and both are two points.

Cohen's $h$, the difference of arcsine-square-root transforms, is stable across
the range. The odds ratio was considered and rejected: it is routinely misread as
a risk ratio by non-specialist readers.

In [10]:
demo = pd.DataFrame([
    {"comparison": "2% -> 4%",   "p1": 0.02, "p2": 0.04},
    {"comparison": "48% -> 50%", "p1": 0.48, "p2": 0.50},
    {"comparison": "85% -> 87%", "p1": 0.85, "p2": 0.87},
])
demo["diff_pp"] = (demo.p2 - demo.p1) * 100
demo["cohens_h"] = [round(abs(sc.cohens_h(a, b)), 3)
                    for a, b in zip(demo.p1, demo.p2)]
demo["ratio"] = (demo.p2 / demo.p1).round(2)
display(demo)
print("Identical in percentage points. Not remotely identical in substance.")
print("This is why the asymmetry result of Part 2 cannot be stated in pp alone.")

,comparison,p1,p2,diff_pp,cohens_h,ratio
0,2% -> 4%,0.02,0.04,2.0,0.119,2.00
1,48% -> 50%,0.48,0.50,2.0,0.040,1.04
2,85% -> 87%,0.85,0.87,2.0,0.058,1.02


Identical in percentage points. Not remotely identical in substance.
This is why the asymmetry result of Part 2 cannot be stated in pp alone.


### 3.4 The free parameters, and how much they matter

The analysis has no fitted weights, but it does contain four choices the data do
not determine. Treating them casually would be the equivalent of tuning on the
test set. Each is set deliberately and its influence measured.

In [11]:
# Free parameter 1: the composite weighting. The only one that materially
# changes a reported quantity. Six schemes, including the three leave-one-out
# variants and two theory-motivated reweightings.
weighting = ms.run_composite_weighting(ind)
display(weighting[["scheme"] + [c for c in weighting.columns if c.startswith("D_")]])

print("Sign of G_s negative in every cell:", weighting.attrs["G_sign_stable"])
print("Sign of D_s negative in every cell:", weighting.attrs["D_sign_stable"])

dcols = [c for c in weighting.columns if c.startswith("D_")]
print(f"\nRange of D_Small across schemes: "
      f"{weighting['D_Small'].min():.2f} to {weighting['D_Small'].max():.2f} pp")
print("=> the DIRECTION is invariant; the POINT ESTIMATE is not.")
print("   This is why the dissertation reports a range, not one decimal place.")

,scheme,D_Micro,D_Small,D_Medium,D_Large
0,"Equal (T,B,P)",-3.35,-13.39,-14.46,-14.74
1,Omit T,-8.53,-17.21,-17.57,-12.80
2,Omit B,-1.33,-15.31,-18.67,-20.99
3,Omit P,-0.17,-7.65,-7.14,-10.45
4,Govern-weighted,-5.42,-14.91,-15.70,-13.97
5,Protect-weighted,-0.75,-11.48,-12.90,-15.72


Sign of G_s negative in every cell: True
Sign of D_s negative in every cell: True

Range of D_Small across schemes: -17.21 to -7.65 pp
=> the DIRECTION is invariant; the POINT ESTIMATE is not.
   This is why the dissertation reports a range, not one decimal place.


In [12]:
# Free parameter 2: the number of clusters. With n = 4, k is bounded above by 3,
# because silhouette is undefined when every point is its own cluster.
from sklearn.preprocessing import StandardScaler

X = ind[["T", "B", "A", "C", "R"]].to_numpy(float)
Xs = StandardScaler().fit_transform(X)

sweep = sc.kmeans_sweep(Xs, k_values=(2, 3), seed=SEED)
display(pd.DataFrame([{k: v for k, v in d.items() if k != "labels"} for d in sweep]))

print("k = 2 selected on silhouette (Rousseeuw 1987), roughly twice that at k = 3.")
print("Inertia is NOT used as the criterion: it falls monotonically in k by")
print("construction, and at n = 4 the elbow heuristic has nothing to work with.")

c:\Users\ifrid\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ifrid\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ifrid\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ifrid\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^

,k,inertia,silhouette,cluster_sizes
0,2,2.823,0.591,"[2, 2]"
1,3,0.856,0.294,"[1, 2, 1]"


k = 2 selected on silhouette (Rousseeuw 1987), roughly twice that at k = 3.
Inertia is NOT used as the criterion: it falls monotonically in k by
construction, and at n = 4 the elbow heuristic has nothing to work with.


In [13]:
# Free parameters 3 and 4: the random seed and the standardisation step.
# Both are measured rather than assumed away.
stability = sc.partition_stability(Xs, k=2, seed=SEED)
print("Leave-one-feature-out: partition recovered in "
      f"{stability['n_stable']}/{stability['n_features']} cases")

from sklearn.cluster import KMeans
base = np.array(sweep[0]["labels"])
moved = sum(
    not (np.array_equal(KMeans(2, n_init=20, random_state=s).fit(Xs).labels_, base)
         or np.array_equal(1 - KMeans(2, n_init=20, random_state=s).fit(Xs).labels_, base))
    for s in range(50))
print(f"Seeds giving a different partition: {moved}/50")

raw = sc.kmeans_sweep(X, k_values=(2,), seed=SEED)[0]["labels"]
same = (np.array_equal(np.array(raw), base) or np.array_equal(1 - np.array(raw), base))
print(f"Partition identical on UNstandardised features: {same}")
print("\n=> Neither the initialisation nor the preprocessing is producing the result.")
print("   What this does NOT establish is that the partition is meaningful.")

c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window

Leave-one-feature-out: partition recovered in 5/5 cases


c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Window

Seeds giving a different partition: 0/50


c:\Users\ifrid\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Partition identical on UNstandardised features: True

=> Neither the initialisation nor the preprocessing is producing the result.
   What this does NOT establish is that the partition is meaningful.


### 3.5 Evaluation metrics

Accuracy, precision, recall and ROC-AUC do not apply: there are no predictions to
score. What has to be evaluated instead is whether each quantity is stated with
the right amount of confidence. Four criteria are used.

1. **Interval coverage** (measured in 3.1). The primary metric, and the only one
   with a formal target.
2. **Attained precision and minimum detectable difference.** A study working from
   published tables cannot choose its sample size, so a prospective power
   calculation would be a fiction. The honest equivalent is to report, after the
   fact, the smallest difference the attained bases could have detected.
3. **Effect size**, reported alongside every test, never instead of one.
4. **Family-wise error rate**, held at 5% across the whole family of contrasts.

Two things are deliberately not measured: out-of-sample generalisation, since
the four bands are the population of bands rather than a draw from it; and
goodness of fit for $C_s$, since $C_s$ is a definitional construct, not a fitted
model, and a fit statistic would imply a validation that has not occurred.

In [14]:
precision = ms.run_precision_budget(ind)
display(precision)

print("Read this table BEFORE reading any result. It states in advance which")
print("questions these data can settle.\n")
print("  - At the small band the minimum detectable difference is 14.1 pp, and")
print("    the differential found there is 13.4 pp: detectable, but only just.")
print("  - At the large band the minimum is 20.4 pp. A differential of 14.7 pp")
print("    reaching significance there reflects the size of the effect, not the")
print("    power of the design.")
print("  - It also explains why the difference-in-differences test in Part 4")
print("    cannot separate the bands: no test on these bases could have.")

,size_band,base,n_eff,rate_pct,moe_pp,min_detectable_diff_pp
0,Micro,All organisations,985,24.8,2.69,5.5
1,Micro,AI-engaged,273,21.5,4.85,9.8
2,Small,All organisations,441,40.7,4.57,9.3
3,Small,AI-engaged,156,27.3,6.93,14.1
4,Medium,All organisations,259,60.0,5.92,12.1
5,Medium,AI-engaged,143,45.6,8.06,16.5
6,Large,All organisations,131,80.5,6.75,13.7
7,Large,AI-engaged,85,65.7,9.89,20.4


Read this table BEFORE reading any result. It states in advance which
questions these data can settle.

  - At the small band the minimum detectable difference is 14.1 pp, and
    the differential found there is 13.4 pp: detectable, but only just.
  - At the large band the minimum is 20.4 pp. A differential of 14.7 pp
    reaching significance there reflects the size of the effect, not the
    power of the design.
  - It also explains why the difference-in-differences test in Part 4
    cannot separate the bands: no test on these bases could have.


---
# Part 4. Results
### (Dissertation Chapter 5)

## RQ1. Sectoral patterns of AI adoption

The headline rate is derived as `1 - (not currently using)`, which is how the
release intends it to be read: the technology columns are not mutually exclusive
and do not sum to the headline. **No base sizes are published in this release, so
no confidence interval or significance test is available for any RQ1 comparison.**
Everything below is descriptive, and is labelled as such.

In [15]:
bics = ld.load_bics_ai_usage(BICS, wave="Wave 147")
mean6 = bics["ai_use_rate"].mean()
all_biz = bics.attrs["all_business_rate"]

t = bics[["sector", "ai_use_rate"]].copy()
t["ai_use_pct"] = (t.pop("ai_use_rate") * 100).round(1)
t["deviation_from_6_sector_mean_pp"] = (t["ai_use_pct"] - mean6 * 100).round(1)
t["vs_all_business_pp"] = (t["ai_use_pct"] - all_biz * 100).round(1)

print(f"All-business rate : {all_biz*100:.1f}%")
print(f"Six-sector mean   : {mean6*100:.1f}%")
print(f"Spread (max-min)  : "
      f"{(bics['ai_use_rate'].max()-bics['ai_use_rate'].min())*100:.1f} pp")
display(t)

All-business rate : 32.6%
Six-sector mean   : 45.5%
Spread (max-min)  : 16.0 pp


,sector,ai_use_pct,deviation_from_6_sector_mean_pp,vs_all_business_pp
0,Life Sciences,51.2,5.7,18.6
1,Creative Industries,49.3,3.8,16.7
2,Advanced Manufacturing,48.3,2.8,15.7
3,Digital and Technologies,45.4,-0.1,12.8
4,Professional and Business Services,43.4,-2.1,10.8
5,Defence,35.2,-10.3,2.6


In [16]:
# Composition matters more than the headline. A sector where half the firms use a
# language model and one where half run bespoke pipelines return the same
# headline rate and are not doing the same thing.
tech = ["ai_llm_rate", "ai_ml_data_rate", "ai_image_ml_rate", "ai_robotics_rate"]
comp = bics.set_index("sector")[tech] * 100
display(comp.round(1))

creative, defence = comp.loc["Creative Industries"], comp.loc["Defence"]
print("Creative Industries : Defence")
for c in tech:
    if np.isfinite(defence[c]) and defence[c] > 0:
        print(f"  {c:20s} {creative[c]/defence[c]:.2f}x   "
              f"(absolute shortfall {creative[c]-defence[c]:+.1f} pp)")
    else:
        print(f"  {c:20s} not computable (suppressed for disclosure control)")

print(f"\nSuppressed cells: {int(comp.isna().sum().sum())} of {comp.size}. "
      "Never imputed.")

,ai_llm_rate,ai_ml_data_rate,ai_image_ml_rate,ai_robotics_rate
sector,,,,
Life Sciences,28.4,19.7,17.2,NaN
Creative Industries,30.1,14.3,8.3,1.1
Advanced Manufacturing,21.6,5.9,11.3,2.5
Digital and Technologies,25.3,15.1,8.9,1.5
Professional and Business Services,24.6,12.1,5.9,NaN
Defence,7.8,2.3,NaN,NaN


Creative Industries : Defence
  ai_llm_rate          3.86x   (absolute shortfall +22.3 pp)
  ai_ml_data_rate      6.22x   (absolute shortfall +12.0 pp)
  ai_image_ml_rate     not computable (suppressed for disclosure control)
  ai_robotics_rate     not computable (suppressed for disclosure control)

Suppressed cells: 4 of 24. Never imputed.


In [17]:
# Sensitivity check: is the Defence result an artefact of deriving the headline
# from the "not currently using" column? The maximum named-technology rate is an
# independent lower bound on adoption of any LISTED technology.
bics["alt_lower_bound"] = bics[tech].max(axis=1)
agree = np.corrcoef(bics["ai_use_rate"].rank(ascending=False),
                    bics["alt_lower_bound"].rank(ascending=False))[0, 1]

print(f"Rank correlation, headline vs alternative construction: {agree:.3f}")
print("Defence lowest under headline measure    :",
      bool(bics.loc[bics.sector == 'Defence', 'ai_use_rate'].iloc[0]
           == bics['ai_use_rate'].min()))
print("Defence lowest under alternative measure :",
      bool(bics.loc[bics.sector == 'Defence', 'alt_lower_bound'].iloc[0]
           == bics['alt_lower_bound'].min()))
print("\n=> The RQ1 ordering is not an artefact of how the headline was derived.")

Rank correlation, headline vs alternative construction: 0.771
Defence lowest under headline measure    : True
Defence lowest under alternative measure : True

=> The RQ1 ordering is not an artefact of how the headline was derived.


## RQ2. Cyber capability by firm size

Wilson intervals on every level; Holm-adjusted contrasts across the family.

In [18]:
rows = []
for _, r in ind.iterrows():
    for key, name in [("T", "Training"), ("B", "Board responsibility"),
                      ("P", "Formal policy"), ("A", "AI adoption"),
                      ("R", "Any breach")]:
        lo, hi = sc.wilson_ci(r[key], r["n_all"])
        rows.append({"size_band": r["size_band"], "indicator": name,
                     "rate_pct": round(r[key]*100, 1),
                     "ci_95": f"[{lo*100:.1f}, {hi*100:.1f}]",
                     "moe_pp": round(sc.moe(r[key], r["n_all"])*100, 2)})
levels = pd.DataFrame(rows)
display(levels.pivot(index="indicator", columns="size_band",
                     values="ci_95")[ld.SIZE_LABELS])

# The composite, with bounding standard errors. The three components are measured
# on the SAME respondents, so their sampling errors are correlated and the
# covariance is not recoverable from published tables. Rather than assume a
# value, both bounds are reported.
print("\nComposite C_s with bounding standard errors:")
for _, r in ind.iterrows():
    lo_se, hi_se = sc.composite_se_bounds([r["T"], r["B"], r["P"]], r["n_all"])
    print(f"  {r['size_band']:7s} C = {r['C']*100:5.2f}%   SE between "
          f"{lo_se*100:.2f} and {hi_se*100:.2f} pp   widest 95% CI "
          f"[{(r['C']-sc.Z_95*hi_se)*100:.1f}, {(r['C']+sc.Z_95*hi_se)*100:.1f}]")

size_band,Micro,Small,Medium,Large
indicator,,,,
AI adoption,"[17.8, 22.8]","[20.5, 28.5]","[33.6, 45.4]","[36.5, 53.3]"
Any breach,"[38.7, 44.8]","[41.7, 51.0]","[58.8, 70.3]","[60.1, 75.8]"
Board responsibility,"[26.1, 31.7]","[32.5, 41.5]","[45.5, 57.6]","[59.6, 75.4]"
Formal policy,"[28.3, 34.1]","[47.5, 56.8]","[69.1, 79.6]","[82.6, 93.3]"
Training,"[12.4, 16.8]","[28.8, 37.6]","[47.7, 59.8]","[77.2, 89.6]"



Composite C_s with bounding standard errors:
  Micro   C = 24.81%   SE between 0.78 and 1.35 pp   widest 95% CI [22.2, 27.4]
  Small   C = 40.71%   SE between 1.33 and 2.31 pp   widest 95% CI [36.2, 45.2]
  Medium  C = 60.04%   SE between 1.72 and 2.97 pp   widest 95% CI [54.2, 65.9]
  Large   C = 80.48%   SE between 1.95 and 3.32 pp   widest 95% CI [74.0, 87.0]


In [19]:
key = mult_detail[mult_detail["label"].str.contains("Micro vs Small|Medium vs Large")]
display(key[["label", "diff_pp", "ci_low_pp", "ci_high_pp", "p_value", "p_holm",
             "cohens_h", "h_magnitude", "reject_holm"]].round(3))

print("What the data support is a TWO-GROUP structure, micro and small on one")
print("side, medium and large on the other, rather than a smooth four-point")
print("gradient. The two contrasts that fail are both micro vs small.")

,label,diff_pp,ci_low_pp,ci_high_pp,p_value,p_holm,cohens_h,h_magnitude,reject_holm
0,Training: Micro vs Small,-18.63,-23.597,-13.801,0.000,0.000,-0.446,small,True
5,Training: Medium vs Large,-30.55,-38.571,-21.207,0.000,0.000,-0.681,medium,True
6,Board responsibility: Micro vs Small,-8.03,-13.384,-2.781,0.003,0.013,-0.171,negligible,True
11,Board responsibility: Medium vs Large,-16.39,-25.940,-6.054,0.002,0.012,-0.336,small,True
12,Formal policy: Micro vs Small,-21.04,-26.452,-15.518,0.000,0.000,-0.430,small,True
17,Formal policy: Medium vs Large,-14.39,-21.454,-6.250,0.001,0.006,-0.381,small,True
18,AI adoption: Micro vs Small,-4.14,-8.982,0.449,0.078,0.313,-0.100,negligible,False
23,AI adoption: Medium vs Large,-5.40,-15.699,4.833,0.306,0.612,-0.109,negligible,False
24,Any breach: Micro vs Small,-4.63,-10.199,0.922,0.103,0.313,-0.093,negligible,False
29,Any breach: Medium vs Large,-3.76,-13.229,6.302,0.459,0.612,-0.080,negligible,False


What the data support is a TWO-GROUP structure, micro and small on one
side, medium and large on the other, rather than a smooth four-point
gradient. The two contrasts that fail are both micro vs small.


## RQ3, step 1. The naive gap $G_s = A_s - C_s$, and why it fails

$A_s$ measures whether a firm has *adopted* a technology. $C_s$ measures whether
it has *governance arrangements*. Their difference has no stable substantive
meaning: a firm could report high $C_s$ and low $A_s$ simply because it has not
adopted AI at all, which says nothing about whether its governance is
proportionate to its AI risk.

It is computed here because the published tables invite the comparison, and
showing why it fails is part of the contribution.

In [20]:
gap = pd.DataFrame({
    "size_band": ind["size_band"],
    "A_pct": (ind["A"]*100).round(1),
    "C_pct": (ind["C"]*100).round(1),
    "G_pp": (ind["G"]*100).round(1),
    "label": np.where(abs(ind["G"]*100) <= 5, "Near balance",
             np.where(ind["G"] < 0, "Cyber ahead", "AI ahead"))})
display(gap)
print("Read at face value: cyber capability leads AI capability at every size,")
print("and the shortfall WIDENS with scale, from -4.7 to -35.8 pp.")
print("This apparently contradicts H1. Step 2 shows why it is an artefact.")

,size_band,A_pct,C_pct,G_pp,label
0,Micro,20.1,24.8,-4.7,Near balance
1,Small,24.3,40.7,-16.4,Cyber ahead
2,Medium,39.3,60.0,-20.7,Cyber ahead
3,Large,44.7,80.5,-35.8,Cyber ahead


Read at face value: cyber capability leads AI capability at every size,
and the shortfall WIDENS with scale, from -4.7 to -35.8 pp.
This apparently contradicts H1. Step 2 shows why it is an artefact.


## RQ3, step 2. The construct-valid differential $D_s = V_s - C_s$

The CSBS asks organisations that use or are considering AI whether they have
*specific cyber security practices in place to manage risks arising from AI*.
Call that rate $V_s$. Both terms of $D_s = V_s - C_s$ are now governance
measures, so $D_s < 0$ has a clear reading.

One asymmetry needs stating. $V_s$ rests on the AI-engaged subpopulation, whose
effective bases are much smaller (273, 156, 143, 85). The intervals reflect that.
And the two populations overlap rather than coincide, so this is a differential
between two rates on distinct populations rather than a within-firm difference.
The direction of that bias is determinable: AI-engaged firms are more
digitally engaged, digital engagement is positively associated with cyber
governance throughout these data, so the $V_s$ subpopulation should show *higher*
general governance than the $C_s$ population. Any negative $D_s$ observed under
those conditions is therefore a *conservative* estimate of the true shortfall.

In [21]:
rows = []
for _, r in ind.iterrows():
    lo, hi = sc.differential_ci(r["V"], r["n_ai"], r["C"], r["n_all"])
    tst = sc.two_proportion_test(r["V"], r["n_ai"], r["C"], r["n_all"])
    ulo, uhi = sc.wilson_ci(r["U"], r["n_ai"])
    rows.append({"size_band": r["size_band"], "n_ai": int(r["n_ai"]),
                 "V_pct": round(r["V"]*100, 1), "C_pct": round(r["C"]*100, 1),
                 "D_pp": round(r["D"]*100, 1),
                 "D_ci_95": f"[{lo:.1f}, {hi:.1f}]",
                 "p_value": round(tst.p_value, 4),
                 "cohens_h": round(tst.cohens_h, 2),
                 "U_pct": round(r["U"]*100, 1),
                 "U_ci_95": f"[{ulo*100:.1f}, {uhi*100:.1f}]"})
diff = pd.DataFrame(rows)
display(diff)

print("Significant at the small, medium and large bands (p = 0.003, 0.005, 0.015).")
print("NOT significant at micro (p = 0.253), where the differential is -3.3 pp.")
print("\nU_s is the finding in the form that matters for practice: the share of")
print("AI-engaged firms operating AI with no risk process around it at all.")

,size_band,n_ai,V_pct,C_pct,D_pp,D_ci_95,p_value,cohens_h,U_pct,U_ci_95
0,Micro,273,21.5,24.8,-3.3,"[-8.6, 2.5]",0.253,-0.08,78.5,"[73.3, 83.0]"
1,Small,156,27.3,40.7,-13.4,"[-21.3, -4.7]",0.003,-0.28,72.7,"[65.2, 79.1]"
2,Medium,143,45.6,60.0,-14.5,"[-24.3, -4.3]",0.005,-0.29,54.4,"[46.2, 62.4]"
3,Large,85,65.7,80.5,-14.7,"[-26.8, -2.8]",0.015,-0.34,34.3,"[25.0, 44.8]"


Significant at the small, medium and large bands (p = 0.003, 0.005, 0.015).
NOT significant at micro (p = 0.253), where the differential is -3.3 pp.

U_s is the finding in the form that matters for practice: the share of
AI-engaged firms operating AI with no risk process around it at all.


## RQ3, step 3. Comparing the two specifications directly

This is the comparison-of-models step. Same composite, same bands, same survey.
The only difference is what sits on the other side of the subtraction.

In [22]:
spec = pd.DataFrame({
    "size_band": ind["size_band"],
    "A_pct": (ind["A"]*100).round(1),
    "V_pct": (ind["V"]*100).round(1),
    "G_pp": (ind["G"]*100).round(1),
    "D_pp": (ind["D"]*100).round(1),
})
spec["divergence_pp"] = (spec["G_pp"] - spec["D_pp"]).abs().round(1)
spec["A_minus_V_pp"] = (spec["A_pct"] - spec["V_pct"]).round(1)
display(spec)

print("The two specifications AGREE where almost nothing is happening (1.4 pp at")
print("micro) and DISAGREE entirely where the evidence is strongest (21.1 pp at")
print("large).\n")
print("Why: A_s is bounded by whether a firm adopted AI at all; V_s is")
print("conditioned on firms that did. At the large band 44.7% report adoption but")
print("65.7% of the AI-engaged report AI-specific practices, so treating the")
print("adoption rate as the capability measure understates AI-side capability by")
print("more than twenty points. G_s does not measure a widening governance")
print("shortfall. It measures the growing distance between an adoption rate and a")
print("governance rate, which is a different quantity and not an interesting one.")

,size_band,A_pct,V_pct,G_pp,D_pp,divergence_pp,A_minus_V_pp
0,Micro,20.1,21.5,-4.7,-3.3,1.4,-1.4
1,Small,24.3,27.3,-16.4,-13.4,3.0,-3.0
2,Medium,39.3,45.6,-20.7,-14.5,6.2,-6.3
3,Large,44.7,65.7,-35.8,-14.7,21.1,-21.0


The two specifications AGREE where almost nothing is happening (1.4 pp at
micro) and DISAGREE entirely where the evidence is strongest (21.1 pp at
large).

Why: A_s is bounded by whether a firm adopted AI at all; V_s is
conditioned on firms that did. At the large band 44.7% report adoption but
65.7% of the AI-engaged report AI-specific practices, so treating the
adoption rate as the capability measure understates AI-side capability by
more than twenty points. G_s does not measure a widening governance
shortfall. It measures the growing distance between an adoption rate and a
governance rate, which is a different quantity and not an interesting one.


In [23]:
# Would the estimator choice of Part 3.1 have changed the D_s conclusion?
cmp_rows = []
for _, r in ind.iterrows():
    nlo, nhi = sc.differential_ci(r["V"], r["n_ai"], r["C"], r["n_all"])
    d = r["V"] - r["C"]
    se = np.sqrt(r["V"]*(1-r["V"])/r["n_ai"] + r["C"]*(1-r["C"])/r["n_all"])
    cmp_rows.append({"size_band": r["size_band"], "D_pp": round(d*100, 1),
                     "newcombe_ci": f"[{nlo:.1f}, {nhi:.1f}]",
                     "wald_ci": f"[{(d-sc.Z_95*se)*100:.1f}, {(d+sc.Z_95*se)*100:.1f}]",
                     "max_abs_diff_pp": round(max(abs(nlo-(d-sc.Z_95*se)*100),
                                                  abs(nhi-(d+sc.Z_95*se)*100)), 2)})
display(pd.DataFrame(cmp_rows))

print("They agree to within 0.4 pp at every band and no conclusion changes.")
print("This is the EXPECTED result and it is reported because it is expected:")
print("D_s is a difference of mid-range rates, and Part 3.1 predicted the")
print("estimator choice would bite at the boundary, not in the middle. Where it")
print("does bite is the exploratory analysis of Part 2, with rates as low as 1.3%.")

,size_band,D_pp,newcombe_ci,wald_ci,max_abs_diff_pp
0,Micro,-3.3,"[-8.6, 2.5]","[-8.9, 2.2]",0.31
1,Small,-13.4,"[-21.3, -4.7]","[-21.7, -5.0]",0.47
2,Medium,-14.5,"[-24.3, -4.3]","[-24.6, -4.3]",0.29
3,Large,-14.7,"[-26.8, -2.8]","[-26.9, -2.6]",0.21


They agree to within 0.4 pp at every band and no conclusion changes.
This is the EXPECTED result and it is reported because it is expected:
D_s is a difference of mid-range rates, and Part 3.1 predicted the
estimator choice would bite at the boundary, not in the middle. Where it
does bite is the exploratory analysis of Part 2, with rates as low as 1.3%.


In [24]:
# Is the differential STABLE above micro, or is that small-sample noise?
sub = diff[diff.size_band != "Micro"].copy()
span = sub["D_pp"].max() - sub["D_pp"].min()
widths_ci = [float(x.strip("[]").split(", ")[1]) - float(x.strip("[]").split(", ")[0])
             for x in sub["D_ci_95"]]
print(f"Spread of D_s across small/medium/large : {span:.1f} pp")
print(f"Widest 95% CI on any single D_s         : {max(widths_ci):.1f} pp")

sm = ind[ind.size_band == "Small"].iloc[0]
lg = ind[ind.size_band == "Large"].iloc[0]
se_sm = np.sqrt(sm.V*(1-sm.V)/sm.n_ai + sm.C*(1-sm.C)/sm.n_all)
se_lg = np.sqrt(lg.V*(1-lg.V)/lg.n_ai + lg.C*(1-lg.C)/lg.n_all)
z = (sm.D - lg.D) / np.sqrt(se_sm**2 + se_lg**2)
from scipy import stats as st
p_did = 2 * (1 - st.norm.cdf(abs(z)))
print(f"\nDifference-in-differences, small vs large: {(sm.D-lg.D)*100:+.1f} pp, "
      f"z = {z:.2f}, p = {p_did:.3f}")

print("\nRead this against the precision budget of Part 3.5. The bands are")
print("indistinguishable because the design LACKS THE PRECISION to distinguish")
print("them, not because they have been shown to be equal: at the large band the")
print("minimum detectable difference is 20.4 pp, so a real narrowing of eight")
print("points would have gone undetected.")
print("\n'Stable' is therefore defensible only in the weak sense. The strong claim")
print("is the negative one, and it is the one H1 needs: there is NO evidence the")
print("differential narrows as general governance improves, and the point")
print("estimates move in the opposite direction.")

Spread of D_s across small/medium/large : 1.3 pp
Widest 95% CI on any single D_s         : 24.0 pp

Difference-in-differences, small vs large: +1.4 pp, z = 0.18, p = 0.857

Read this against the precision budget of Part 3.5. The bands are
indistinguishable because the design LACKS THE PRECISION to distinguish
them, not because they have been shown to be equal: at the large band the
minimum detectable difference is 20.4 pp, so a real narrowing of eight
points would have gone undetected.

'Stable' is therefore defensible only in the weak sense. The strong claim
is the negative one, and it is the one H1 needs: there is NO evidence the
differential narrows as general governance improves, and the point
estimates move in the opposite direction.


In [25]:
# Exploratory clustering. Reported for completeness, weighted accordingly.
km_labels = sweep[0]["labels"]
print("Cluster assignment:", dict(zip(ld.SIZE_LABELS, km_labels)))
print(f"Silhouette at k=2 : {sweep[0]['silhouette']:.3f}")
print("\nWith four observations and k = 2, any partition separates the data along")
print("whatever axis dominates the variance, and here that axis is firm size")
print("itself. This CANNOT demonstrate a typology and is not treated as doing so.")
print("It answers a narrower question the gap table cannot: whether the five")
print("indicators group the bands consistently when considered JOINTLY.")

Cluster assignment: {'Micro': 0, 'Small': 0, 'Medium': 1, 'Large': 1}
Silhouette at k=2 : 0.591

With four observations and k = 2, any partition separates the data along
whatever axis dominates the variance, and here that axis is firm size
itself. This CANNOT demonstrate a typology and is not treated as doing so.
It answers a narrower question the gap table cannot: whether the five
indicators group the bands consistently when considered JOINTLY.


---
# Part 5. Validation
### (Dissertation Section 3.8)

Cross-validation and a held-out test set are unavailable with four observations.
Their absence does not remove the obligation to validate; it changes the
instruments. Five checks, each targeting a distinct way a result here could be an
artefact rather than a finding.

In [26]:
# CHECK 1. Do the numbers in the dissertation match the source spreadsheets?
# Both analysis scripts write their machine-readable outputs to out/:
# run_analysis.py produces results.json, model_selection.py produces
# model_selection.json. Both are run here so the notebook regenerates every
# output it later reads, starting from a clean checkout.
import subprocess

res = subprocess.run([sys.executable, "run_analysis.py"], cwd="analysis",
                     capture_output=True, text=True)
tail = [l for l in res.stdout.split("\n") if "reproduce" in l or "discrep" in l.lower()]
print("\n".join(tail) if tail else res.stdout[-600:])

sel = subprocess.run([sys.executable, "model_selection.py"], cwd="analysis",
                     capture_output=True, text=True)
print(f"\nmodel_selection.py exit {sel.returncode}: out/model_selection.json written")
if sel.returncode:
    print(sel.stderr[-800:])


33 of 33 quoted values reproduce to within 0.06 pp (rounding tolerance).
No discrepancies.

model_selection.py exit 0: out/model_selection.json written


In [27]:
# CHECK 2. Sensitivity to indicator construction (full detail).
robust = weighting.copy()
gcols = [c for c in robust.columns if c.startswith("G_")]
display(robust[["scheme"] + gcols])

print("Sign negative in every cell:", bool((robust[gcols] < 0).all().all()))
print("\nOne classification DOES move, and it is reported rather than buried:")
print(f"  Dropping training takes G_Micro from {robust.loc[0,'G_Micro']:.2f} to "
      f"{robust.loc[1,'G_Micro']:.2f} pp, crossing the delta = 5 threshold and")
print("  reclassifying micro firms from 'near balance' to 'cyber ahead'.")
print("  The near-balance label is therefore a boundary artefact, not a property")
print("  of micro firms.")

,scheme,G_Micro,G_Small,G_Medium,G_Large
0,"Equal (T,B,P)",-4.67,-16.43,-20.72,-35.76
1,Omit T,-9.85,-20.25,-23.83,-33.82
2,Omit B,-2.65,-18.35,-24.93,-42.01
3,Omit P,-1.49,-10.69,-13.40,-31.47
4,Govern-weighted,-6.74,-17.95,-21.96,-34.99
5,Protect-weighted,-2.07,-14.52,-19.16,-36.73


Sign negative in every cell: True

One classification DOES move, and it is reported rather than buried:
  Dropping training takes G_Micro from -4.67 to -9.85 pp, crossing the delta = 5 threshold and
  reclassifying micro firms from 'near balance' to 'cyber ahead'.
  The near-balance label is therefore a boundary artefact, not a property
  of micro firms.


In [28]:
# CHECKS 3 and 4 were run in Part 3.4 (partition stability) and Part 4
# (alternative construction of the BICS headline). Restated here for the record.
print(f"CHECK 3  partition stable under leave-one-feature-out : "
      f"{stability['n_stable']}/{stability['n_features']}")
print(f"         partition stable under 50 random seeds       : {50-moved}/50")
print(f"CHECK 4  BICS ordering invariant to headline construction: "
      f"rank correlation {agree:.3f}")

print("\nCHECK 5  adversarial reading of the central result")
print("  Challenge A: V_s and C_s rest on different populations, so a negative")
print("               D_s could be composition rather than a within-firm gap.")
print("  Response  : the direction of that bias is determinable and runs the")
print("               other way, so D_s UNDERSTATES the shortfall. See Part 4.")
print("  Challenge B: breach prevalence rises with capability, contradicting a")
print("               protective effect.")
print("  Response  : detection confound, established descriptively in Part 2.")
print("               Consequence: R_s is not used as an outcome anywhere.")

CHECK 3  partition stable under leave-one-feature-out : 5/5
         partition stable under 50 random seeds       : 50/50
CHECK 4  BICS ordering invariant to headline construction: rank correlation 0.771

CHECK 5  adversarial reading of the central result
  Challenge A: V_s and C_s rest on different populations, so a negative
               D_s could be composition rather than a within-firm gap.
  Response  : the direction of that bias is determinable and runs the
               other way, so D_s UNDERSTATES the shortfall. See Part 4.
  Challenge B: breach prevalence rises with capability, contradicting a
               protective effect.
  Response  : detection confound, established descriptively in Part 2.
               Consequence: R_s is not used as an outcome anywhere.


---
# Part 6. Findings and implications
### (Dissertation Chapters 5-6)

## The four findings

**1. AI adoption varies substantially by sector, and its composition varies more
than its headline.** Adoption runs from 35.2% in Defence to 51.2% in Life
Sciences. Defence uses large language models at roughly a quarter of the rate
seen in Creative Industries. Descriptive only: the release publishes no bases.

**2. The SME cyber deficit is asymmetric, not general.** Smaller firms hold
technical protective controls at roughly two-thirds to four-fifths of the
large-firm rate, and governance, response, assurance and training capabilities at
roughly a fifth to a half. Meanwhile the incidents they actually suffer are
overwhelmingly human-mediated. Investment is concentrated where the remedy is a
purchase; the exposure is concentrated where the remedy is a practice.

**3. The comparison the published tables invite is construct-invalid.** Setting
the AI adoption rate against a cyber governance composite yields a shortfall that
appears to widen from -4.7 to -35.8 pp with firm size. That is not a governance
finding. It is the growing distance between an adoption rate and a governance
rate.

**4. The like-for-like comparison reverses the diagnosis.** Among firms already
using or considering AI, AI-specific cyber practices sit 13 to 15 pp below their
general cyber governance from the small band upwards, significantly so at every
band above micro. **78.5% of AI-engaged micro firms report no AI-specific
practices at all.** The differential does not narrow as general governance
improves.

## What follows

**For the hypotheses.** H1 is supported in its central component: the shortfall
is common, and it does not close as general governance improves. The second half
of H1, concerning breach outcomes, is *not testable* with these data, because
the detection confound makes breach prevalence unusable as an outcome variable.
H2 survives only weakly: the indicators do cohere by scale, but these data cannot
separate genuine complementarity from common dependence on firm size.

**For practice.** The diagnosis matters more than it might appear. A general
governance deficit calls for building governance capacity, which is expensive and
slow. An AI-specific extension gap calls for extending an existing policy to a new
technology, which is neither. Roughly three-quarters of AI-engaged small firms are
running AI with no risk process around it, and not because they lack governance
capacity: they have built it, and have not pointed it at the new problem.

**For theory.** Domain-specific governance does not follow automatically from
general governance capacity. The organisations here with the strongest general
cyber governance still show a differential of roughly fifteen points. Governance
extends to a new technology through a deliberate act, not through spillover.

**What this study cannot settle.** Whether practitioners recognise this account of
their own organisations; whether firms with AI and without AI-specific practices
regard risk management as the vendor's responsibility; whether the Defence
adoption pattern reflects capability or permission. These are questions about
organisational reasoning, and no aggregate table can answer them.

In [29]:
# Regenerate every figure used in the dissertation.
os.system(f"{sys.executable} analysis/make_figures.py")

0

In [30]:
# Final provenance stamp: everything reported above, in one machine-readable file.
with open("out/results.json") as fh:
    results = json.load(fh)
with open("out/model_selection.json") as fh:
    selection = json.load(fh)

print("out/results.json         keys:", list(results.keys()))
print("out/model_selection.json keys:", list(selection.keys()))
print(f"\nVerification: {len(results['verification']['discrepancies'])} discrepancies "
      "between the dissertation text and the source spreadsheets.")
print("\nEvery number in the dissertation is generated from these two files.")
print("None was transcribed by hand.")

out/results.json         keys: ['rq1', 'rq2', 'rq3', 'verification']
out/model_selection.json keys: ['seed', 'n_simulations', 'anchors', 'interval_widths', 'coverage', 'multiplicity', 'multiplicity_summary', 'kmeans_sweep', 'partition_stability', 'scaling_effect', 'composite_weighting', 'composite_weighting_G_sign_stable', 'composite_weighting_D_sign_stable', 'precision_budget']

Verification: 0 discrepancies between the dissertation text and the source spreadsheets.

Every number in the dissertation is generated from these two files.
None was transcribed by hand.
